## Parse and design TCR library example notebook:
To call and count TCRαβ amino acid clonotypes, filtered_contig_annotations.csv generated by CellRanger are parsed in this example notebook using `parse_10x_contig_list`. Briefly, contigs are filtered on Cell Ranger's high confidence, is_cell, full-length, and productive flags (`filter_on_quality=True`). Barcodes with only a single chain (`remove_single_chain_barcodes=True`), or with multiple chains that were all alpha or all beta but not both (`remove_only_alpha_or_beta_chain_barcodes=True`), are removed. Dual chain filtering is based on the ratio of the non-top chain's UMI count to the top chain's UMI count (`dual_chain_umi_count_ratio_threshold`, if `remove_dual_alpha_and_dual_beta_barcodes=False`, `remove_dual_beta_barcodes=False`, `remove_dual_alpha_barcodes=False` otherwise dual chains are always filtered). For each 10X dataset, a non-stringent ratio threshold is set by inspecting the density-normalized histogram of non-top chain UMI count ratios, choosing a cutoff just below the point where non-top alpha chains are less frequent than non-top beta chains, as dual alpha chains are generated more frequently than dual beta chains. Dual TCR chains that pass filtering are then ordered deterministically by UMI count when building the clonotype string, to prevent incorrectly counting dual chain clonotypes twice. Amino acid clonotypes are defined by TRAV, TRAJ, TRBV, and TRBJ gene usage and the CDR3α and CDR3β amino acid sequences. Amino acid clonotype counts and frequencies are calculated separately for CD8+ and CD4+ T cells of each sample by counting of cell barcodes (`calculate_clonotype_frequencies`). To create TCRαβ clonotypes for screening from cells expressing dual alpha and/ or dual beta chains, cell barcodes were expanded into multiple rows that cover the two (one dual chain) or four (two dual chains) possible combination (before in `parse_10x_contig_list`), with the counts of the parent dual chain clonotype being used for the newly generated single pair clonotypes. For TCRαβ library design, the resulting unique amino acid clonotypes are sorted by their clonotype frequency (`clonotype_aa` frequency), and within each `donor_col` group only the first most frequent occurrence of each duplicate unique single pair amino acid clonotype `unique_clonotype_aa` is kept (duplicate single pair `unique_clonotype_aa` clonotypes can have different frequencies if these originate from different dual chain clonotypes in `clonotype_aa`) using `drop_clonotype_duplicates`, and the required number of top ranked clonotypes can be selected for screening and prepared for [T-RAP TCR assembly](https://doi.org/10.1101/2025.04.28.651095).

In [ ]:
from pathlib import Path
import pandas as pd

from tcr_toolbox.tcr_parsing.parser_utils import calculate_clonotype_frequencies, drop_clonotype_duplicates
from tcr_toolbox.tcr_parsing.tcr_10x_contig_parser import filter_meta_10x_df_on_tcr_umi_counts, parse_10x_contig_list, merge_combined_meta_df_with_tcr_contig_dfs
from tcr_toolbox.utils.plot_utils import plot_10x_tcr_umi_counts_hist, plot_clonotype_frequency

In [ ]:
seq_run_dir = Path("/tcr_assembly_datasets/patient-1-4")
outs_dir = seq_run_dir / "outs"

In [3]:
outs_dir.mkdir(parents=False, exist_ok=False)

In [ ]:
dual_chain_umi_count_ratio_threshold = 0.4
gem_10x_run_name_list = ["patient01", "patient02", "patient03", "patient04"]
run_dir_list = [seq_run_dir / "cellranger_runs" / gem_10x_run for gem_10x_run in gem_10x_run_name_list]
contig_file_list = [gem_10x_run_dir / "outs" / "per_sample_outs" / gem_10x_run_dir.name / "vdj_t" / "filtered_contig_annotations.csv" for gem_10x_run_dir in run_dir_list]

In [ ]:
combined_contig_df = parse_10x_contig_list(
    gem_10x_run_name_list=gem_10x_run_name_list,
    contig_file_list=contig_file_list,
    outs_dir=outs_dir,
    meta_data_column_list=None,
    filter_on_quality=True,
    keep_highest_read_only=False,
    remove_dual_alpha_and_dual_beta_barcodes=False,
    remove_dual_beta_barcodes=False,
    remove_dual_alpha_barcodes=False,
    remove_single_chain_barcodes=True,
    remove_only_alpha_or_beta_chain_barcodes=True,
    dual_chain_umi_count_ratio_threshold=dual_chain_umi_count_ratio_threshold,
    threads=8,
)

combined_contig_df.to_csv(outs_dir / "combined_contig_df.csv")

In [ ]:
meta_csv = seq_run_dir / "seurat_or_scanpy_meta_with_cd8_and_cd4_calls.csv"
combined_meta_df = pd.read_csv(meta_csv)

In [ ]:
combined_meta_df = merge_combined_meta_df_with_tcr_contig_dfs(combined_contig_df=combined_contig_df, combined_meta_df=combined_meta_df)

In [8]:
umi_count_quantile_threshold = 0.025

In [ ]:
plot_10x_tcr_umi_counts_hist(
    combined_meta_df=combined_meta_df,
    save_dir=outs_dir,
    fig_name="before_filtering",
    bin_max=70,
    bin_size=1,
    ylim_max=0.25,
    umi_count_quantile_threshold=umi_count_quantile_threshold,
)

In [ ]:
combined_meta_df = filter_meta_10x_df_on_tcr_umi_counts(meta_10x_df=combined_meta_df, sample_col="sample_run", umi_count_quantile_threshold=umi_count_quantile_threshold)

Shape combined_meta_df before UMI count quantile filtering: 8220
Shape combined_meta_df after UMI count quantile filtering: 7612


In [ ]:
plot_10x_tcr_umi_counts_hist(
    combined_meta_df=combined_meta_df,
    save_dir=outs_dir,
    fig_name="after_filtering",
    bin_max=70,
    bin_size=1,
    ylim_max=0.25,
    umi_count_quantile_threshold=umi_count_quantile_threshold,
)

In [ ]:
# To calculate within patient donor and CD8+ or CD4+ T cells, set donor_col = "patient_coreceptor" instead of "patient"
# donor_col="patient" (pooled): larger, more stable denominator, and CD4/CD8 clonotypes stay on one comparable scale.
# donor_col="patient_coreceptor" (split): safer if clonotype calls overlap CD4/CD8 (mis-calls, doublets, DP cells),
# since pooling would let a few mislabeled cells affect clonotype ranking for the *other* coreceptor.
combined_meta_df["patient_coreceptor"] = combined_meta_df["patient"] + "_" + combined_meta_df["cd4_cd8_call"]

combined_meta_df = calculate_clonotype_frequencies(
    tcr_meta_df=combined_meta_df,
    donor_col="patient", # or patient_coreceptor
    clonotype_nt_col="clonotype_nt",
    clonotype_aa_col="clonotype_aa",
    clonotype_count_nt_col="clonotype_count_nt",
    clonotype_count_aa_col="clonotype_count_aa",
    clonotype_freq_nt_col="clonotype_frequency_nt",
    clonotype_freq_aa_col="clonotype_frequency_aa",
)

In [ ]:
combined_meta_df = drop_clonotype_duplicates(
    tcr_meta_df=combined_meta_df,
    also_drop_clonotype_beta_aa_duplicates=False,
    donor_col="patient",
    clonotype_freq_col="clonotype_frequency_aa",
    clonotype_col="unique_clonotype_aa",
    # clonotype_beta_col = "unique_clonotype_beta_aa",
    clonotype_shared_with_col="shared_with",
)

Number of TCRs before dropping duplicate clonotypes: 7612
Number of TCRs after dropping duplicate clonotypes: 5098


In [ ]:
combined_meta_df["group"] = "patient-1-4"
combined_meta_df["library"] = combined_meta_df["patient"] + "_" + combined_meta_df["cd4_cd8_call"]
combined_meta_df["custom_name"] = (
    combined_meta_df["patient"] + "_" +
    (combined_meta_df.groupby("library").cumcount() + 1).astype(str)
)
combined_meta_df.to_csv(outs_dir / "combined_meta_contig_dropped_clonotype_duplicates_df.csv")

In [ ]:
plot_clonotype_frequency(
    combined_meta_df=combined_meta_df,
    donor_col="patient",
    outs_dir=outs_dir,
    ylim_max=800,
    clonotype_type="aa",
    clonotype_col="unique_clonotype_aa",
    clonotype_freq_col="clonotype_frequency_aa",
    bin_max=0.08,
    bin_size=0.001,
    coreceptor_col="cd4_cd8_call",
    CD8_str="CD8",
    CD4_str="CD4",
)

In [ ]:
# For example, select CD8 T cells for assembly
combined_meta_df = combined_meta_df.loc[combined_meta_df["cd4_cd8_call"] == "CD8", :]
combined_meta_df.reset_index(drop = True, inplace = True)
combined_meta_df.to_csv(outs_dir / "combined_meta_contig_df_dropped_clonotype_duplicates_CD8.csv")

## Output files and columns written by this pipeline

**`outs/parse_10x_from_contig.log`** - parsing log for each `filtered_contig_annotation.csv`. The following barcode occurrences are counted (barcodes with more than 4 contigs are not counted, as these are not considered for parsing):

- `single_chain`: barcode has only 1 detected chain (whether alpha or beta).
- `only_alpha_or_beta_chains`: barcode has only a single chain type detected (whether 1-4 chains) and not at least 1 detected alpha and beta pair.
- `single_pair`: barcode has 1 detected alpha and beta pair. Not counted under its own key in the log; reflected in the final `detected_chains` distribution printed at the end.
- `3 rows`: barcode has 3 detected chains.
- `dual_alpha`: barcode has 3 detected chains of which 2 chains are alpha chains and 1 chain is a beta chain (i.e., dual_alpha).
- `dual_beta`: barcode has 3 detected chains of which 2 chains are beta chains and 1 chain is an alpha chain (i.e., dual_beta).
- `nt_duplicate_dual_alpha_or_nt_duplicate_dual_beta`: of those 3 chains, the seemingly duplicated alpha or beta rows are identical nt clonotypes rather than a true dual pair (so they collapse to 1 unique alpha/beta clonotype, not 2). The barcode cannot be resolved into a dual_alpha or dual_beta pair, so it is dropped.
- `4 rows`: barcode has 4 detected chains.
- `dual_alpha_and_dual_beta`: barcode has 4 detected chains of which 2 are alpha chains and 2 are beta chains (i.e., dual_alpha_and_dual_beta). Barcodes with 4 rows that have e.g., 3 alpha chains are removed (counted as `4_rows_due_to_3_alpha_or_beta`).
- `umi_count_ratio_dual_alpha_removed`: the number of non-top UMI count alpha chains removed that have a lower UMI count fraction than `dual_chain_umi_count_ratio_threshold` of the top UMI count alpha chain (in dual_alpha barcodes), thereby converting the barcode to single_pair.
- `umi_count_ratio_dual_beta_removed`: the number of non-top UMI count beta chains removed that have a lower UMI count fraction than `dual_chain_umi_count_ratio_threshold` of the top UMI count beta chain (in dual_beta barcodes), thereby converting the barcode to single_pair.
- `umi_count_ratio_dual_alpha_removed_from_dual_alpha_and_dual_beta`: the number of non-top UMI count alpha chains removed that have a lower UMI count fraction than `dual_chain_umi_count_ratio_threshold` of the top UMI count alpha chain (in dual_alpha_and_dual_beta barcodes). If both the alpha and beta chain are removed, the barcode is converted to single_pair. If only the alpha chain is removed, the barcode is converted to dual_beta.
- `umi_count_ratio_dual_beta_removed_from_dual_alpha_and_dual_beta`: the number of non-top UMI count beta chains removed that have a lower UMI count fraction than `dual_chain_umi_count_ratio_threshold` of the top UMI count beta chain (in dual_alpha_and_dual_beta barcodes). If both the alpha and beta chain are removed, the barcode is converted to single_pair. If only the beta chain is removed, the barcode is converted to dual_alpha.

**`outs/umi_count_ratio_least_abundant_most_abundant_dual_chain_hist.pdf`** - for each barcode with 2 detected alpha and/or beta chains (i.e., dual_alpha, dual_beta, dual_alpha_and_dual_beta), the UMI count ratio is calculated as the UMI count of the non-top chain (i.e., least abundant) divided by that of the top chain (i.e., most abundant). This PDF shows a histogram of the resulting UMI count ratios for alpha and beta chains separately. In addition, the `dual_chain_umi_count_ratio_threshold` used to filter out lower-confidence chains (i.e., those with lower UMI support relative to the higher-confidence chain with more UMI support within the same barcode) is visualized.

**`outs/before_filtering_umi_counts_histogram.pdf`** - alpha chain + beta chain UMI counts histogram before alpha + beta chain UMI count quantile filtering using the `umi_count_quantile_threshold`. Yellow line indicates the `umi_count_quantile_threshold` that was used.

**`outs/after_filtering_umi_counts_histogram.pdf`** - alpha chain + beta chain UMI counts histogram after alpha + beta UMI count quantile filtering.

**`outs/combined_contig_df.csv`** - all parsed `filtered_contig_annotation.csv`s combined into a single dataframe, without `clonotype_aa` dropping and before merging with cell metadata (that merge happens afterward, in memory, via `merge_combined_meta_df_with_tcr_contig_dfs`). Each row stores a unique alpha + beta pair, and the parsing pipeline adds the following columns:

- `clonotype_nt`: nucleotide clonotype string of the alpha + beta pair. Defined by TRAV/TRAJ/TRBV/TRBJ gene usage and the CDR3 alpha and CDR3 beta nucleotide sequences. Dual TCR chains that passed dual chain filtering are ordered deterministically by UMI count when building the `clonotype_nt` string, to prevent incorrectly counting dual chain clonotypes twice. For single_pair barcodes, `clonotype_nt` equals `unique_clonotype_nt`. To create single_pair clonotypes for screening, `clonotype_nt` is expanded into 2 rows for dual_alpha/dual_beta barcodes, each with its own `unique_clonotype_nt` entry for every possible single_pair combination that can be screened. For dual_alpha_and_dual_beta barcodes, `clonotype_nt` is expanded into 4 rows (one row per combination), again each with its own `unique_clonotype_nt`. Clonotype frequency is then estimated by counting `clonotype_nt`, not `unique_clonotype_nt`, so each dual-chain barcode (the actual clone) is counted once rather than once per expanded single_pair combination, and that parental dual chain count is copied onto every one of that barcode's duplicate rows.
- `clonotype_aa`: same as `clonotype_nt` but with CDR3 alpha and beta amino acid sequences instead of nucleotide sequences.
- `detected_chains`: whether barcode is `single_pair`, `dual_alpha`, `dual_beta`, or `dual_alpha_and_dual_beta` (see the `parse_10x_from_contig.log` description above for definitions).
- `dual_chain_umi_count_ratio`: for each barcode with 2 detected alpha and/or beta chains (i.e., dual_alpha, dual_beta, dual_alpha_and_dual_beta), the UMI count ratio calculated as the UMI count of the non-top chain (i.e., least abundant) divided by that of the top chain (i.e., most abundant).
- `unique_clonotype_nt`: single_pair nucleotide clonotypes, unique within cell barcodes (see explanation for `clonotype_nt` above).
- `unique_clonotype_aa`: single_pair amino acid clonotypes, unique within cell barcodes. Since unique single_pair amino acid clonotypes are functionally screened, `unique_clonotype_aa` is later made unique for selecting single_pair clonotypes for screening (see below). 
- `clonotype_alpha_nt`: alpha clonotype part of `clonotype_nt`.
- `clonotype_beta_nt`: beta clonotype part of `clonotype_nt`.
- `clonotype_alpha_aa`: alpha clonotype part of `clonotype_aa`.
- `clonotype_beta_aa`: beta clonotype part of `clonotype_aa`.
- `unique_clonotype_alpha_nt`: alpha clonotype part of `unique_clonotype_nt`.
- `unique_clonotype_beta_nt`: beta clonotype part of `unique_clonotype_nt`.
- `unique_clonotype_alpha_aa`: alpha clonotype part of `unique_clonotype_aa`.
- `unique_clonotype_beta_aa`: beta clonotype part of `unique_clonotype_aa`.

**`outs/combined_meta_contig_dropped_clonotype_duplicates_df.csv`** - To allow selection of the top single_pair clonotypes for screening, made by: 
1. Merging combined_contig_df with a Seurat/Scanpy-like meta dataframe containing at least a donor group and CD8_vs_CD4 call columns. 
2. Counting cell barcodes for each unique `clonotype_aa`. 
3. Sorting the rows by their `clonotype_frequency_aa`. 
4. Keeping only the first most frequent occurrence of each duplicate `unique_clonotype_aa` single_pair (duplicate `unique_clonotype_aa` can have different frequencies if these originate from different `clonotype_aa` dual chain clonotypes). 

And contains the following additional columns: 

- `clonotype_count_nt`: `clonotype_nt` count, calculated by counting of cell barcodes. Using `clonotype_nt` and not `unique_clonotype_nt` means each dual chain clonotype is correctly counted once, by its parent dual chain clonotype, rather than once per expanded single_pair combination.
- `clonotype_frequency_nt`: `clonotype_nt` barcode frequency, calculated within donor group. 
- `clonotype_count_aa`: `clonotype_aa` count, calculated by counting of cell barcodes. Using `clonotype_aa` and not `unique_clonotype_aa` means each dual chain clonotype is correctly counted once, by its parent dual chain clonotype, rather than once per expanded single_pair combination.
- `clonotype_frequency_aa`: `clonotype_aa` barcode frequency, calculated within donor group. 
- `shared_with`: if a `unique_clonotype_aa` is shared across more than one donor group, stores the sorted list of *all* donors carrying that clonotype (including this row's own donor) joined by `-`. It is `None` if the clonotype is only found in one donor. This column is potentially useful for selecting TCRs, as `unique_clonotype_aa` sharing between donors may be indicative of public clonotype expansion.

**`outs/clonotype_frequency/`** - folder containing `clonotype_frequency_aa` histogram PDFs for each donor.